[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/74_optimal_subsequence_solution.ipynb)

# Solution: Optimal Subsequence (GCD Chain)

Reference solution — binary search + prime factorisation + DP.


## 解析

**结论：二分答案 + 质因数分解 + DP。**

要求“最大元素的最小可能取值”，对这个最大值 `x` **二分**。可行性单调：若 `x` 可行，则更大的 `x` 一定可行（可选元素只增不减，原子序列仍合法）。

固定 `x` 后只保留 `a_i <= x` 的元素，问题变成：**在这些元素中是否存在长度 `>= k` 的子序列，使相邻两数 `gcd > 1`**。两数能相邻当且仅当它们**共享某个质因子**。于是把每个数分解成不同质因子，做如下 DP：

- `dp_i` = 以第 `i` 个元素结尾的最长合法子序列长度；
- 维护 `best[p]` = 当前已遍历元素中、含质因子 `p` 作结尾的最大 `dp`；
- `dp_i = 1 + max(best[p] for p | a_i)`，若无可连接前驱则 `dp_i = 1`；
- 算完 `dp_i` 后用它更新所有 `best[p]`；出现 `dp_i >= k` 即可行。

**边界**：`a_i = 1` 没有质因子，只能单独成长度 1；`k = 1` 时答案一定是数组最小值。

**复杂度**：设 `ω(a_i)` 为不同质因子个数（`a_i <= 1e9` 时最多约 9）。单次判定 `O(Σ ω(a_i))`，二分 `O(log n)` 次，总 `O(log n · Σ ω(a_i))`。下面的参考实现用试除法分解质因子（在本题小数据上足够快）；面对 `1e9` 且 `n` 很大的极端数据，可换成 Miller–Rabin + Pollard–Rho 加速分解。


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
from typing import List


In [ ]:
# ✅ SOLUTION

def _prime_factors(x: int):
    """distinct prime factors via trial division (Pollard-Rho for huge inputs)"""
    fs = set()
    d = 2
    while d * d <= x:
        if x % d == 0:
            fs.add(d)
            while x % d == 0:
                x //= d
        d += 1
    if x > 1:
        fs.add(x)
    return fs

class Solution:
    def min_max_element(self, a: List[int], k: int) -> int:
        fac = [_prime_factors(x) for x in a]   # factor each number once

        def feasible(limit: int) -> bool:
            best = {}
            for i, x in enumerate(a):
                if x > limit:
                    continue
                cur = 1                        # element alone = chain of length 1
                for p in fac[i]:
                    cur = max(cur, best.get(p, 0) + 1)
                if cur >= k:
                    return True
                for p in fac[i]:
                    if cur > best.get(p, 0):
                        best[p] = cur
            return False

        vals = sorted(set(a))
        lo, hi, ans = 0, len(vals) - 1, -1
        while lo <= hi:                        # binary search the smallest feasible max
            mid = (lo + hi) // 2
            if feasible(vals[mid]):
                ans = vals[mid]
                hi = mid - 1
            else:
                lo = mid + 1
        return ans


In [ ]:
# Demo
sol = Solution()
print(sol.min_max_element([2, 4, 3, 9, 6], 3))     # 6
print(sol.min_max_element([5, 7, 11, 13, 17], 2))  # -1


In [ ]:
from torch_judge import check
check('optimal_subsequence')
